Passo 1: Bloco de importação de bibliotecas:

In [1]:
import pandas as pd
import numpy as np
import random
import os
import re
import json
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta

RF01: Gerar e salvar o dataset:

In [5]:
def gerar_dataset_ecommerce(n_registros=300, seed=42):
    """Gera um dataset sintético de e-commerce brasileiro com dados
    propositalmente sujos: valores nulos, datas inválidas e strings
    com espaços extras — para praticar limpeza no RF03."""
    random.seed(seed)
    np.random.seed(seed)

    produtos = [
        'Tênis Esportivo', 'Smartphone Samsung', 'Notebook Dell',
        'Fone Bluetooth', 'Camiseta Algodão', 'Calça Jeans',
        'Mochila Escolar', 'Relógio Digital', 'Perfume Importado',
        'Cafeteira Elétrica'
    ]
    categorias = {
        'Tênis Esportivo':    'Calçados',
        'Smartphone Samsung': 'Eletrônicos',
        'Notebook Dell':      'Eletrônicos',
        'Fone Bluetooth':     'Eletrônicos',
        'Camiseta Algodão':   'Vestuário',
        'Calça Jeans':        'Vestuário',
        'Mochila Escolar':    'Acessórios',
        'Relógio Digital':    'Acessórios',
        'Perfume Importado':  'Beleza',
        'Cafeteira Elétrica': 'Eletrodomésticos'
    }
    precos = {
        'Tênis Esportivo':    320,
        'Smartphone Samsung': 1850,
        'Notebook Dell':      3200,
        'Fone Bluetooth':     210,
        'Camiseta Algodão':   79,
        'Calça Jeans':        149,
        'Mochila Escolar':    130,
        'Relógio Digital':    280,
        'Perfume Importado':  390,
        'Cafeteira Elétrica': 250
    }
    regioes = ['Sudeste', 'Sul', 'Nordeste', 'Centro-Oeste', 'Norte']
    estados = {
        'Sudeste':      ['SP', 'RJ', 'MG', 'ES'],
        'Sul':          ['PR', 'SC', 'RS'],
        'Nordeste':     ['BA', 'PE', 'CE', 'MA', 'RN'],
        'Centro-Oeste': ['GO', 'MT', 'MS', 'DF'],
        'Norte':        ['AM', 'PA', 'RO', 'AC']
    }
    pagamentos = ['Cartão de Crédito', 'Boleto', 'Pix', 'Cartão de Débito']
    clientes = [f"Cliente_{i:03d}" for i in range(1, 51)]
    data_inicio = datetime(2024, 1, 1)

    dados = []
    for i in range(n_registros):
        produto  = random.choice(produtos)
        regiao   = random.choice(regioes)
        estado   = random.choice(estados[regiao])
        qtd      = random.randint(1, 5)
        preco    = precos[produto]
        data     = data_inicio + timedelta(days=random.randint(0, 364))
        cliente  = random.choice(clientes)
        pagamento = random.choice(pagamentos)

        # Dados intencionalmente sujos para limpeza
        if random.random() < 0.05:
            qtd = None                      # valor nulo
        if random.random() < 0.04:
            preco = None                    # valor nulo
        if random.random() < 0.03:
            produto = "  " + produto        # espaço extra (string suja)
        if random.random() < 0.03:
            cliente = cliente + "  "        # espaço extra no final
        if random.random() < 0.02:
            pagamento = None                # pagamento nulo

        data_str = (data.strftime("%Y-%m-%d")
                    if random.random() > 0.02 else "DATA INVALIDA")

        dados.append({
            "id_pedido":        i + 1,
            "data_pedido":      data_str,
            "cliente":          cliente,
            "produto":          produto,
            "categoria":        categorias.get(produto.strip(), "Outros"),
            "regiao":           regiao,
            "estado":           estado,
            "quantidade":       qtd,
            "preco_unitario":   preco,
            "forma_pagamento":  pagamento,
        })

    return pd.DataFrame(dados)


# Gerar e salvar
df_bruto = gerar_dataset_ecommerce()
os.makedirs("../data/raw", exist_ok=True)
df_bruto.to_csv("../data/raw/vendas.csv", index=False)

print(f"Dataset gerado com {len(df_bruto)} registros e {len(df_bruto.columns)} colunas.")
print(f"Colunas: {list(df_bruto.columns)}")
df_bruto.head()

Dataset gerado com 300 registros e 10 colunas.
Colunas: ['id_pedido', 'data_pedido', 'cliente', 'produto', 'categoria', 'regiao', 'estado', 'quantidade', 'preco_unitario', 'forma_pagamento']


,id_pedido,data_pedido,cliente,produto,categoria,regiao,estado,quantidade,preco_unitario,forma_pagamento
0,1,2024-04-24,Cliente_009,Smartphone Samsung,Eletrônicos,Sudeste,MG,2.0,1850.0,Cartão de Crédito
1,2,2024-04-11,Cliente_046,Perfume Importado,Beleza,Norte,AM,5.0,390.0,Cartão de Débito
2,3,2024-06-21,Cliente_007,Calça Jeans,Vestuário,Nordeste,PE,2.0,149.0,Cartão de Crédito
3,4,2024-05-30,Cliente_041,Smartphone Samsung,Eletrônicos,Centro-Oeste,GO,5.0,1850.0,Pix
4,5,2024-08-20,Cliente_041,Fone Bluetooth,Eletrônicos,Sudeste,ES,3.0,210.0,Pix


RF02: Inspecionar os dados:

In [7]:
def inspecionar_dados(df):
    """Exibe informações básicas do DataFrame:
    shape, colunas, tipos, nulos e estatísticas descritivas."""
    print("\n=== INSPEÇÃO INICIAL DO DATASET ===")
    print(f"Shape: {df.shape}")
    print(f"\nColunas: {list(df.columns)}")
    print(f"\nTipos de dados:\n{df.dtypes}")
    print(f"\nValores nulos por coluna:\n{df.isnull().sum()}")
    print(f"\nPrimeiros registros:\n{df.head()}")
    print(f"\nEstatísticas descritivas:\n{df.describe()}")
    return df.describe(include="all")


inspecionar_dados(df_bruto)


=== INSPEÇÃO INICIAL DO DATASET ===
Shape: (300, 10)

Colunas: ['id_pedido', 'data_pedido', 'cliente', 'produto', 'categoria', 'regiao', 'estado', 'quantidade', 'preco_unitario', 'forma_pagamento']

Tipos de dados:
id_pedido            int64
data_pedido            str
cliente                str
produto                str
categoria              str
regiao                 str
estado                 str
quantidade         float64
preco_unitario     float64
forma_pagamento        str
dtype: object

Valores nulos por coluna:
id_pedido           0
data_pedido         0
cliente             0
produto             0
categoria           0
regiao              0
estado              0
quantidade          8
preco_unitario      8
forma_pagamento    10
dtype: int64

Primeiros registros:
   id_pedido data_pedido        cliente             produto    categoria  \
0          1  2024-04-24    Cliente_009  Smartphone Samsung  Eletrônicos   
1          2  2024-04-11  Cliente_046     Perfume Importado       

,id_pedido,data_pedido,cliente,produto,categoria,regiao,estado,quantidade,preco_unitario,forma_pagamento
count,300.000000,300,300,300,300,300,300,292.000000,292.000000,290
unique,NaN,206,60,16,6,5,20,NaN,NaN,4
top,NaN,DATA INVALIDA,Cliente_007,Notebook Dell,Eletrônicos,Sul,PR,NaN,NaN,Pix
freq,NaN,6,12,40,95,68,29,NaN,NaN,82
mean,150.500000,NaN,NaN,NaN,NaN,NaN,NaN,2.904110,792.969178,NaN
std,86.746758,NaN,NaN,NaN,NaN,NaN,NaN,1.373929,1072.918377,NaN
min,1.000000,NaN,NaN,NaN,NaN,NaN,NaN,1.000000,79.000000,NaN
25%,75.750000,NaN,NaN,NaN,NaN,NaN,NaN,2.000000,149.000000,NaN
50%,150.500000,NaN,NaN,NaN,NaN,NaN,NaN,3.000000,280.000000,NaN
75%,225.250000,NaN,NaN,NaN,NaN,NaN,NaN,4.000000,390.000000,NaN
